# Gemma 4 E4B + LoRA: русско-тувинский перевод (Colab, A100)

Обучение на **всём train** из `prepare_data.py` — тот же сплит, что у NLLB v3,
тот же **чистый тест (1369 пар)**, те же промпты `ru→tyv:` / `tyv→ru:`.
Эталон NLLB v3: **49.2 / 49.1** chrF++.

## Как пользоваться — всегда одинаково: «Выполнить все»

Ноутбук сам понимает, что делать:
- **адаптера на Диске нет** → обучает (ячейка 8), потом оценивает;
- **адаптер уже обучен** → пропускает обучение, загружает адаптер и сразу оценивает;
- **обучение прервалось** → продолжает с последнего чекпойнта.

Ничего пропускать вручную не нужно. Переобучить с нуля — `FORCE_RETRAIN = True` в ячейке 5.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → **A100** + **High-RAM**;
слева 🔑 Secrets → `HF_TOKEN` (токен Write) → включить «Notebook access».

**Во время долгих ячеек** раз в 10–15 минут кликай в ноутбуке — иначе Colab Pro отключит сессию «за бездействие».

Первый раз на новой модели: `SMOKE_TEST = True` (60 шагов, ~10 минут) → убедиться, что всё работает → `False`.

История запусков и результаты — в `EXPERIMENTS.md` проекта.

In [ ]:
# Ячейка 1 — GPU и библиотеки
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip uninstall -y -q torchao
!pip install -q -U transformers peft trl accelerate datasets sacrebleu
import torch, transformers, peft, trl
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
print("bf16:", torch.cuda.is_bf16_supported())
# Старый torchao в Colab ломает новый peft — он удаляется выше. Если peft уже был импортирован
# до удаления, нужен один перезапуск сеанса.

In [ ]:
# Ячейка 2 — Google Диск (чекпойнты и адаптер переживут отключение Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ячейка 3 — вход в Hugging Face (секрет HF_TOKEN, токен с правом Write)
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Вошёл через Colab Secrets")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# Ячейка 4 — данные: ТОТ ЖЕ prepare_data.py, что на Маке (сплит NLLB v3 + чистый тест)
# После отключения Colab файлы в /content пропадают — ячейка попросит загрузить prepare_data.py снова.
import os
if not os.path.exists("prepare_data.py"):
    from google.colab import files
    files.upload()
if not os.path.exists("data/train.jsonl"):
    !python prepare_data.py --train-limit 0 --out data
!wc -l data/*.jsonl

In [ ]:
# Ячейка 5 — настройки
SMOKE_TEST    = False   # True — пробный прогон на 60 шагов (для новой модели или новых настроек)
FORCE_RETRAIN = False   # True — обучить заново, даже если адаптер уже есть на Диске

MODEL_ID  = "google/gemma-4-E4B-it"
RUN_NAME  = "gemma4-e4b-tyv-lora"
DRIVE_DIR = f"/content/drive/MyDrive/tyv-llm/{RUN_NAME}"
HUB_REPO  = "Agisight/tyv-gemma4-e4b"

LORA_R, LORA_ALPHA = 32, 64
LR          = 2e-4
BATCH       = 64        # A100 80 ГБ: ~39 ГБ; скорость упирается в GPU, больше батч не ускоряет
MAX_LEN     = 256
EPOCHS      = 1
SAVE_STEPS  = 1000
EVAL_STEPS  = 1000

if SMOKE_TEST:
    DRIVE_DIR += "-smoke"
os.makedirs(DRIVE_DIR, exist_ok=True)
ADAPTER_DIR = f"{DRIVE_DIR}/final_adapter"
TRAINED = os.path.exists(f"{ADAPTER_DIR}/adapter_config.json") and not FORCE_RETRAIN
print("Папка:", DRIVE_DIR)
print("Режим:", "адаптер уже обучен → только оценка" if TRAINED else "обучение")

In [ ]:
# Ячейка 6 — модель: новая LoRA (для обучения) или готовый адаптер с Диска
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda")

if TRAINED:
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    print("Загружен обученный адаптер:", ADAPTER_DIR)
else:
    # Только обычные Linear-слои языковой модели. У Gemma 4 есть зрение и звук со слоями
    # Gemma4ClippableLinear, которые LoRA не поддерживает, — их пропускаем.
    TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
    SKIP = ("vision", "audio", "image", "multi_modal", "embed_vision", "embed_audio")
    target_names = [n for n, m in base.named_modules()
                    if type(m) is torch.nn.Linear and n.split(".")[-1] in TARGETS
                    and not any(s in n for s in SKIP)]
    print(f"LoRA-слоёв: {len(target_names)}; пример: {target_names[:2]}")
    base.config.use_cache = False
    lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
                      target_modules=target_names, task_type="CAUSAL_LM")
    model = get_peft_model(base, lora)
    model.print_trainable_parameters()

In [ ]:
# Ячейка 7 — датасет: chat-формат → prompt/completion (loss только по переводу)
if not TRAINED:
    from datasets import load_dataset
    to_pc = lambda ex: {"prompt": ex["messages"][:-1], "completion": ex["messages"][-1:]}
    train = load_dataset("json", data_files="data/train.jsonl", split="train").map(to_pc, remove_columns=["messages"])
    valid = load_dataset("json", data_files="data/valid.jsonl", split="train").map(to_pc, remove_columns=["messages"])
    if SMOKE_TEST:
        train = train.select(range(2000))
    valid = valid.select(range(min(500, len(valid))))
    print("train:", len(train), "| valid:", len(valid))
    print(train[0])
else:
    print("Пропуск: адаптер уже обучен")

In [ ]:
# Ячейка 8 — обучение (пропускается, если адаптер уже есть; продолжает с чекпойнта при обрыве)
# Полный прогон Gemma 4 E4B на A100: 8494 шага, ~7.5 часа.
if not TRAINED:
    import glob, math
    from trl import SFTConfig, SFTTrainer
    total_steps = 60 if SMOKE_TEST else math.ceil(len(train) / BATCH) * EPOCHS
    warmup = max(1, int(0.03 * total_steps))
    print(f"шагов всего: {total_steps}, разогрев: {warmup}")
    cfg = SFTConfig(
        output_dir=DRIVE_DIR,
        num_train_epochs=EPOCHS,
        max_steps=60 if SMOKE_TEST else -1,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=warmup,                 # warmup_ratio в новой transformers убран
        bf16=True,
        max_length=MAX_LEN,
        completion_only_loss=True,
        # Токенизация в 8 процессов: ~2 мин вместо ~12.
        # НЕ ПРОВЕРЕНО в деле (добавлено 2026-09-30 после первого обучения, которое шло без этой строки).
        # Если ячейка упадёт на «Tokenizing train dataset» — удали строку ниже, остальное проверено.
        dataset_num_proc=8,
        logging_steps=10 if SMOKE_TEST else 50,
        eval_strategy="steps",
        eval_steps=30 if SMOKE_TEST else EVAL_STEPS,
        save_strategy="steps",
        save_steps=30 if SMOKE_TEST else SAVE_STEPS,
        save_total_limit=2,
        report_to="none",
        dataloader_num_workers=2,
    )
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train, eval_dataset=valid, processing_class=tok)
    last = sorted(glob.glob(f"{DRIVE_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
    print("Продолжаю с", last[-1] if last else "начала")
    trainer.train(resume_from_checkpoint=last[-1] if last else None)
    trainer.save_model(ADAPTER_DIR)
    print("Адаптер сохранён:", ADAPTER_DIR)
else:
    print("Пропуск обучения: адаптер уже есть →", ADAPTER_DIR)

In [ ]:
# Ячейка 9 — оценка chrF++ на ЧИСТОМ тесте (как evaluate.py на Маке)
# Переводы сохраняются на Диск: eval_results.json — их потом можно прогнать через
# metrics_extra.py и comet_eval.py на Маке.
import json, re, torch
from sacrebleu.metrics import CHRF, BLEU
from prepare_data import PROMPTS

EVAL_LIMIT = 200 if SMOKE_TEST else 100000   # примеров на направление (полный тест = 1369)
tok.padding_side = "left"
model.eval(); model.config.use_cache = True

rows = [json.loads(l) for l in open("data/eval_pairs.jsonl", encoding="utf-8")]
chrf, bleu = CHRF(word_order=2), BLEU()
results = []
for d in ("ru-tyv", "tyv-ru"):
    items = [r for r in rows if r["direction"] == d][:EVAL_LIMIT]
    hyps = []
    for i in range(0, len(items), 64):
        batch = items[i:i+64]
        prompts = [tok.apply_chat_template([{"role": "user", "content": PROMPTS[d].format(src=r["src"])}],
                                           add_generation_prompt=True, tokenize=False) for r in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            # то же ограничение длины, что у NLLB: 32 + 3 × длина входа
            out = model.generate(**enc, max_new_tokens=min(256, 32 + 3 * enc.input_ids.shape[1]), do_sample=False)
        for o in out[:, enc.input_ids.shape[1]:]:
            h = tok.decode(o, skip_special_tokens=True)
            hyps.append(re.sub(r"<think>.*?</think>", "", h, flags=re.S).strip())
        print(f"  {d}: {min(i + 64, len(items))}/{len(items)}")
    refs = [r["ref"] for r in items]
    results += [{**r, "hyp": h} for r, h in zip(items, hyps)]
    print(f"{d}: chrF++ = {chrf.corpus_score(hyps, [refs]).score:.1f}  "
          f"BLEU = {bleu.corpus_score(hyps, [refs]).score:.1f}  (n={len(items)})")
    for r, h in list(zip(items, hyps))[:3]:
        print("   src:", r["src"], "\n   hyp:", h, "\n   ref:", r["ref"])

json.dump(results, open(f"{DRIVE_DIR}/eval_results.json", "w"), ensure_ascii=False)
print("Переводы:", f"{DRIVE_DIR}/eval_results.json")
print("Эталон NLLB v3 на чистом тесте: ru→tyv 49.2 | tyv→ru 49.1")

In [ ]:
# Ячейка 9б — оценка на ПОЛНОМ тесте NLLB (1999 пар, с дублями train) — для сравнения с карточкой NLLB
# Чистый тест (ячейка 9) — честная проверка; полный — сопоставим с цифрами NLLB v3 из карточки (48.5 / 48.1).
import json, re, torch
from sacrebleu.metrics import CHRF, BLEU
from prepare_data import PROMPTS

if not os.path.exists("data_full/eval_pairs.jsonl"):
    !python prepare_data.py --train-limit 0 --keep-dup-test --out data_full
rows = [json.loads(l) for l in open("data_full/eval_pairs.jsonl", encoding="utf-8")]
if SMOKE_TEST:
    rows = rows[:400]
chrf, bleu = CHRF(word_order=2), BLEU()
results_full = []
for d in ("ru-tyv", "tyv-ru"):
    items = [r for r in rows if r["direction"] == d]
    hyps = []
    for i in range(0, len(items), 64):
        batch = items[i:i+64]
        prompts = [tok.apply_chat_template([{"role": "user", "content": PROMPTS[d].format(src=r["src"])}],
                                           add_generation_prompt=True, tokenize=False) for r in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=min(256, 32 + 3 * enc.input_ids.shape[1]), do_sample=False)
        for o in out[:, enc.input_ids.shape[1]:]:
            hyps.append(re.sub(r"<think>.*?</think>", "", tok.decode(o, skip_special_tokens=True), flags=re.S).strip())
        print(f"  {d}: {min(i + 64, len(items))}/{len(items)}")
    refs = [r["ref"] for r in items]
    results_full += [{**r, "hyp": h} for r, h in zip(items, hyps)]
    print(f"{d}: chrF++ = {chrf.corpus_score(hyps, [refs]).score:.1f}  "
          f"BLEU = {bleu.corpus_score(hyps, [refs]).score:.1f}  (n={len(items)})")
json.dump(results_full, open(f"{DRIVE_DIR}/eval_results_full.json", "w"), ensure_ascii=False)
print("Переводы:", f"{DRIVE_DIR}/eval_results_full.json")
print("NLLB v3 на полном тесте (карточка): ru→tyv 48.5 | tyv→ru 48.1")

In [ ]:
# Ячейка 10 — публикация на Hugging Face (запускать вручную, когда качество устраивает)
# PUSH_ADAPTER: только LoRA-адаптер (~280 МБ) — быстро, достаточно для воспроизведения.
# PUSH_MERGED: слитая 16-битная модель (~16 ГБ) — нужна для mlx_lm.convert на Маке.
PUSH_ADAPTER = True
PUSH_MERGED  = False
assert not SMOKE_TEST, "Публиковать только полный прогон"

if PUSH_ADAPTER:
    model.push_to_hub(HUB_REPO + "-lora", private=True)
    tok.push_to_hub(HUB_REPO + "-lora", private=True)
    print("Адаптер:", f"https://huggingface.co/{HUB_REPO}-lora")

if PUSH_MERGED:
    merged = model.merge_and_unload()
    merged.config.use_cache = True
    SAVE_DIR = f"{DRIVE_DIR}/merged_bf16"
    merged.save_pretrained(SAVE_DIR, safe_serialization=True)
    tok.save_pretrained(SAVE_DIR)
    merged.push_to_hub(HUB_REPO, private=True)
    tok.push_to_hub(HUB_REPO, private=True)
    print("Слитая модель:", f"https://huggingface.co/{HUB_REPO}")

In [ ]:
# Ячейка 11 — слить адаптер с моделью и сконвертировать в MLX (4 и 8 бит) прямо в Colab
# Зачем в Colab: конвертация Gemma 4 требует десятки ГБ ОЗУ (тут 167 ГБ), а на Маке 16 ГБ и мало диска.
# На Мак потом скачивается готовая MLX-модель (~5 ГБ для 4 бит).
# Конвертер — mlx-vlm (официальные MLX-версии Gemma 4 сделаны им). Ранние версии ломали
# квантизацию PLE-слоёв Gemma 4 (модель выдавала мусор) — поэтому ставим свежую и проверяем перевод.
# Запускать вручную, когда нужна MLX-версия. Занимает ~20–40 минут.
CONVERT_BITS = (4, 8)
assert not SMOKE_TEST, "Конвертировать только полный прогон"

!pip install -q -U "mlx[cpu]" mlx-vlm
import gc, torch
from huggingface_hub import HfApi

MERGED = "/content/merged_bf16"
merged = model.merge_and_unload()
merged.config.use_cache = True
merged.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
try:
    from transformers import AutoProcessor
    AutoProcessor.from_pretrained(MODEL_ID).save_pretrained(MERGED)   # mlx-vlm ждёт файлы процессора
except Exception as e:
    print("processor:", str(e)[:200])
del merged, model, base
gc.collect(); torch.cuda.empty_cache()
!du -sh {MERGED}

api = HfApi()
for bits in CONVERT_BITS:
    out = f"/content/tyv-gemma4-e4b-mlx-{bits}bit"
    !python -m mlx_vlm.convert --hf-path {MERGED} --mlx-path {out} -q --q-bits {bits}
    !du -sh {out}
    # быстрая проверка: перевод должен быть осмысленным, а не мусором
    !python -m mlx_vlm.generate --model {out} --prompt "ru→tyv: Завтра я поеду в Кызыл к родителям." --max-tokens 40 --temperature 0.0
    repo = f"{HUB_REPO}-mlx-{bits}bit"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(repo_id=repo, folder_path=out, commit_message=f"Gemma 4 E4B tyv LoRA merged, MLX {bits}-bit")
    print("Загружено:", f"https://huggingface.co/{repo}")
print("Модель в памяти удалена. Для ячеек 9–10 перезапусти ячейку 6.")